# Classical Machine Learning Models for Protein Function Prediction
This notebook trains Logistic Regression and SVM on the three pre‑processed datasets.

## Prerequisites
Run the preprocessing notebook (`protein_function_prediction.ipynb`) **first** so that the `data/processed/*.npy` files exist.

## 1. Imports & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import joblib
import time

sns.set_style("whitegrid")

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

ensure_dir("outputs/models")
ensure_dir("outputs/figures")

## 2. Load Pre‑processed Data

In [ ]:
# Load saved arrays – labels are saved as object arrays, so use allow_pickle=True
X1 = np.load('data/processed/X1.npy')
y1_raw = np.load('data/processed/y1.npy', allow_pickle=True)

X2 = np.load('data/processed/X2.npy')
y2_raw = np.load('data/processed/y2.npy', allow_pickle=True)

X3 = np.load('data/processed/X3.npy')
y3_raw = np.load('data/processed/y3.npy', allow_pickle=True)

print("Dataset shapes:")
print(f"X1: {X1.shape}, y1: {y1_raw.shape}")
print(f"X2: {X2.shape}, y2: {y2_raw.shape}")
print(f"X3: {X3.shape}, y3: {y3_raw.shape}")

## 3. Training & Evaluation Helper Function

In [ ]:
def run_experiment(X, y_raw, dataset_name, output_prefix):
    """
    Splits data, scales features, tunes Logistic Regression and SVM,
    evaluates, saves models, figures, and returns metrics.
    """
    # Encode labels
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    class_names = le.classes_
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"\n{'='*50}")
    print(f"{dataset_name}")
    print(f"  Classes: {class_names}")
    print(f"  Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
    
    # Save label encoder
    joblib.dump(le, f"outputs/models/{output_prefix}_label_encoder.pkl")
    
    # Base pipeline with scaling (important for SVM and LR)
    lr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(random_state=42, class_weight='balanced', max_iter=2000))
    ])
    svm_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', LinearSVC(random_state=42, class_weight='balanced', dual=False, max_iter=2000))
    ])
    
    # Hyperparameter grids
    param_grid_lr = {'lr__C': [0.1, 1, 10], 'lr__solver': ['lbfgs', 'saga']}
    param_grid_svm = {'svc__C': [0.1, 1, 10]}
    
    results = []
    
    # --- Logistic Regression ---
    print("\n--- Logistic Regression ---")
    lr_grid = GridSearchCV(lr_pipe, param_grid_lr, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
    lr_grid.fit(X_train, y_train)
    best_lr = lr_grid.best_estimator_
    print(f"Best params: {lr_grid.best_params_}")
    
    t0 = time.time()
    y_pred_lr = best_lr.predict(X_test)
    pred_time_lr = time.time() - t0
    
    lr_metrics = {
        'accuracy': accuracy_score(y_test, y_pred_lr),
        'precision': precision_score(y_test, y_pred_lr, average='macro', zero_division=0),
        'recall': recall_score(y_test, y_pred_lr, average='macro'),
        'f1': f1_score(y_test, y_pred_lr, average='macro'),
        'train_time': lr_grid.refit_time_ if hasattr(lr_grid, 'refit_time_') else None,
        'pred_time': pred_time_lr
    }
    print(f"Accuracy: {lr_metrics['accuracy']:.4f} | F1: {lr_metrics['f1']:.4f}")
    
    # Confusion matrix
    cm_lr = confusion_matrix(y_test, y_pred_lr)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_lr, annot=False, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{dataset_name} - Logistic Regression')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'outputs/figures/{output_prefix}_logreg_cm.png', dpi=150)
    plt.show()
    
    joblib.dump(best_lr, f'outputs/models/{output_prefix}_logreg.pkl')
    results.append({'dataset': dataset_name, 'model': 'LogisticRegression', **lr_metrics})
    
    # --- SVM ---
    print("\n--- SVM ---")
    svm_grid = GridSearchCV(svm_pipe, param_grid_svm, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
    svm_grid.fit(X_train, y_train)
    best_svm = svm_grid.best_estimator_
    print(f"Best params: {svm_grid.best_params_}")
    
    t0 = time.time()
    y_pred_svm = best_svm.predict(X_test)
    pred_time_svm = time.time() - t0
    
    svm_metrics = {
        'accuracy': accuracy_score(y_test, y_pred_svm),
        'precision': precision_score(y_test, y_pred_svm, average='macro', zero_division=0),
        'recall': recall_score(y_test, y_pred_svm, average='macro'),
        'f1': f1_score(y_test, y_pred_svm, average='macro'),
        'train_time': svm_grid.refit_time_ if hasattr(svm_grid, 'refit_time_') else None,
        'pred_time': pred_time_svm
    }
    print(f"Accuracy: {svm_metrics['accuracy']:.4f} | F1: {svm_metrics['f1']:.4f}")
    
    cm_svm = confusion_matrix(y_test, y_pred_svm)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_svm, annot=False, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{dataset_name} - SVM')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'outputs/figures/{output_prefix}_svm_cm.png', dpi=150)
    plt.show()
    
    joblib.dump(best_svm, f'outputs/models/{output_prefix}_svm.pkl')
    results.append({'dataset': dataset_name, 'model': 'SVM', **svm_metrics})
    
    return results

## 4. Run All Experiments

In [ ]:
all_results = []

# Dataset 1
all_results += run_experiment(X1, y1_raw, 'Dataset1', 'ds1')

# Dataset 2
all_results += run_experiment(X2, y2_raw, 'Dataset2', 'ds2')

# Dataset 3
all_results += run_experiment(X3, y3_raw, 'Dataset3', 'ds3')

## 5. Compile and Compare Results

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv('outputs/results.csv', index=False)
print("\nFinal results table:")
results_df

In [ ]:
# Comparison plot
metrics = ['accuracy', 'precision', 'recall', 'f1']
melted = results_df.melt(id_vars=['dataset', 'model'], value_vars=metrics,
                         var_name='metric', value_name='score')

plt.figure(figsize=(12, 6))
sns.barplot(data=melted, x='metric', y='score', hue='model')
plt.title('Model Performance Comparison Across Datasets')
plt.ylim(0, 1)
plt.legend(title='Model')
plt.tight_layout()
plt.savefig('outputs/figures/model_comparison.png', dpi=150)
plt.show()

All trained models, confusion matrices, and `results.csv` are now saved in the `outputs/` folder.